# Afternoon class 30/08 — Worksheet 12 SOLUTIONS: chunked reading   (L03)

Every cell below was executed in the lab image (pandas 3.0.5) against the real
1,093-row `data/big_sales.csv`, and the quoted output is what it actually
printed.

Question 5 is the one to re-read, and it is the most important result in the
whole class. Two correct totals of the same column disagree.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Worksheet 12 — Chunked reading. Run this once.
import pandas as pd

# The full file: every one of the 1,093 orders, not the 300-row sample.
PATH = "data/big_sales.csv"

whole = pd.read_csv(PATH)
print("whole file:", whole.shape)
print("Sales column dtype:", whole["Sales"].dtype)

PART A — the mechanics

### Question 1

`type(reader)` -> **`TextFileReader`**, not a DataFrame. -> 5 chunks: `250, 250, 250, 250, 93`.

Creating the reader read nothing. It is a lazy iterator that pulls the
next batch of rows each time round the loop, which is the entire point:
peak memory is one chunk, not one file.

That also means the file handle stays open until the iterator is exhausted,
and that you get exactly one pass over it — Q10.

In [ ]:
reader = pd.read_csv(PATH, chunksize=250)
print("type:", type(reader).__name__)
print()
n = 0
for chunk in reader:
    n += 1
    print("chunk %d -> %s" % (n, chunk.shape))
print()
print("chunks:", n)

### Question 2

`[250, 250, 250, 250, 93]`, summing to `1093`. -> `1093 / 250 = 4.372`.

Four full chunks and a remainder of 93. `chunksize` is a maximum, not a
guarantee, and the last chunk is whatever is left.

Code that assumes every chunk is the same size — computing an average as
`total / (n_chunks * chunksize)`, say — is wrong by the size of that
remainder and will be wrong by a different amount on every file.

In [ ]:
sizes = [len(c) for c in pd.read_csv(PATH, chunksize=250)]
print("chunk sizes:", sizes)
print("sum:", sum(sizes))
print("rows in file:", len(whole))
print()
print("1093 / 250 =", 1093 / 250)

### Question 3

Counted in chunks `1093`, read in one go `1093`. -> they agree.

Counting rows is *associative*: adding integers in any grouping gives the
same answer. Hold on to that, because the next question does the same thing
with floats and gets a different result.

In [ ]:
total_rows = 0
for chunk in pd.read_csv(PATH, chunksize=250):
    total_rows += len(chunk)
print("counted in chunks:", total_rows)
print("read in one go:   ", len(whole))
print("agree:", total_rows == len(whole))

PART B — the same sum, two ways

### Question 4

Chunked total -> `np.float64(1605576.2175)`.

The deck's exact pattern: a running total, one chunk at a time, never
holding more than 250 rows in memory. It works and the number looks clean.

In [ ]:
total = 0
for chunk in pd.read_csv(PATH, chunksize=250):
    total += chunk["Sales"].sum()
print("chunked total:", repr(total))

### Question 5

chunked `1605576.2175` vs one-shot **`1605576.2174999998`**. -> `equal: False`, difference `2.3283064365386963e-10`.

**Same file, same column, same addition, two different answers.** Nothing
was filtered, nothing was missing, and neither number is a bug.

Floating-point addition is not associative. `(a + b) + c` and
`a + (b + c)` can differ in the last bits, because each intermediate result
is rounded to fit in 64 bits. Chunking changes the grouping of the
additions, so it changes which roundings happen. The one-shot sum groups
them differently again — Pandas uses a pairwise summation internally rather
than a naive left-to-right loop.

The practical consequences are the ones to take away:

1. **Never test two float totals with `==`.** This is how a validation
   step that compares a streamed total against a batch total fails nightly
   for a difference of `2.3283064365386963e-10`.
2. **A reconciliation that must balance exactly should not be done in
   floats.** Use integers — count in cents — or `Decimal`.
3. **Reporting either number to the penny is fine.** See Q7.

The same phenomenon appeared in the SQL sessions when per-group totals did
not sum to the overall total. It is not a Pandas quirk; it is IEEE-754, and
it is under every tool you will ever use.

In [ ]:
total = 0
for chunk in pd.read_csv(PATH, chunksize=250):
    total += chunk["Sales"].sum()

one_shot = whole["Sales"].sum()

print("chunked: ", repr(total))
print("one shot:", repr(one_shot))
print()
print("equal:", total == one_shot)
print("difference:", repr(total - one_shot))

### Question 6

as stored `1605576.2174999998`, ascending `1605576.2175`, descending `1605576.2175`.

No chunking anywhere here — one column, three orderings, two different
answers. That isolates the cause: it is the **order of addition**, not the
reading strategy.

Sorting first happened to reproduce the chunked answer, which is a
coincidence of this data and not a rule. Do not read this as 'sort before
summing for accuracy' — the genuinely more accurate approaches are pairwise
or Kahan summation, and `.sum()` already does the former.

What it does prove is that the difference in Q5 has nothing to do with
chunking being wrong. Chunking is fine. Floats are just like this.

In [ ]:
plain = whole["Sales"].sum()
sorted_sum = whole["Sales"].sort_values().sum()
reversed_sum = whole["Sales"].sort_values(ascending=False).sum()

print("as stored:  ", repr(plain))
print("ascending:  ", repr(sorted_sum))
print("descending: ", repr(reversed_sum))
print()
print("stored == ascending:", plain == sorted_sum)

### Question 7

Both round to `1605576.22`. -> absolute difference `2.3283064365386963e-10`, relative difference `1.450137596185898e-16`.

The relative error is `1.450137596185898e-16`, essentially one unit in the
last place of a 64-bit float — the smallest difference representable at
this magnitude. You cannot do better without changing number types.

Two tenths of a **nanodollar** on 1.6 million. Rounded to cents the two
agree exactly, so for every reporting purpose these numbers are identical.

That is the balanced conclusion: the difference is real and must not be
tested with `==`, and it is also far too small to affect any decision made
from the total. Both halves matter — people who learn the first half
sometimes start distrusting arithmetic entirely, which is its own kind of
wrong.

In [ ]:
total = 0
for chunk in pd.read_csv(PATH, chunksize=250):
    total += chunk["Sales"].sum()
one_shot = whole["Sales"].sum()

print("chunked  (2dp):", round(total, 2))
print("one shot (2dp):", round(one_shot, 2))
print()
print("absolute difference:", repr(abs(total - one_shot)))
print("relative difference:", abs(total - one_shot) / one_shot)

PART C — filtering while you stream

### Question 8

Streamed `(156, 10)`, direct `(156, 10)`. -> the same 156 OrderIDs.

Filtering inside the loop and concatenating the survivors gives exactly
the result of filtering the whole frame — because *selection* is order
independent, unlike float summation.

This is the pattern the deck flags as 'appropriate': the filter removes
most rows, so the concatenated result is far smaller than the input. The
'risky' version it warns about is appending every chunk unfiltered, which
reassembles the whole file in memory and undoes the point of chunking.

In [ ]:
pieces = []
for chunk in pd.read_csv(PATH, chunksize=250):
    pieces.append(chunk[chunk["Sales"] > 3000])

big = pd.concat(pieces, ignore_index=True)
direct = whole[whole["Sales"] > 3000]

print("streamed:", big.shape)
print("direct:  ", direct.shape)
print("same rows:", sorted(big["OrderID"]) == sorted(direct["OrderID"]))

### Question 9

Without `ignore_index` -> labels `[1, 25, 31, 41, 47, 63, ...]`; **still unique** (`True`). With it -> `[0, 1, 2, ...]`. -> `kept.loc[0]` raises `KeyError: 0`; `renumbered.loc[0]` gives OrderID `5251`.

The index is unique — chunks cover disjoint row ranges, so no label can
repeat. So this is not the duplicate-label problem from worksheet 03.

It is a *contiguity* problem. The surviving labels are the original row
numbers of the matching rows, so they start at 1, skip most values, and
have gaps everywhere. `kept.loc[0]` raises because row 0 did not pass the
filter, and any code assuming `.loc[0]` reaches the first row is broken.

Which version you want depends on the question. Keep the original labels
when you need to trace a row back to its position in the source file. Use
`ignore_index=True` when the result is a new dataset in its own right and
the old row numbers are meaningless — which is usually the case after a
filter.

In [ ]:
pieces = []
for chunk in pd.read_csv(PATH, chunksize=250):
    pieces.append(chunk[chunk["Sales"] > 3000])

kept = pd.concat(pieces)
renumbered = pd.concat(pieces, ignore_index=True)

print("without ignore_index, first 12 labels:", list(kept.index[:12]))
print("with ignore_index,    first 12 labels:", list(renumbered.index[:12]))
print()
print("still unique without ignore_index:", kept.index.is_unique)

# Unique, but not contiguous -- and it does not start at 0.
try:
    print(kept.loc[0])
except Exception as exc:
    print("kept.loc[0] -> %s: %s" % (type(exc).__name__, exc))
print("renumbered.loc[0] OrderID:", renumbered.loc[0, "OrderID"])

### Question 10

After consuming all 5 chunks, `next(reader)` -> **raises** `StopIteration`.

The reader is exhausted. It is a one-pass iterator over an open file, not
a re-readable collection, so a second traversal yields nothing.

This is why every question above created a fresh `pd.read_csv(...,
chunksize=...)` rather than reusing one. It is a real trap in longer
scripts: compute a total in one loop, then loop again to filter, and the
second loop silently does nothing — a `for` loop over an exhausted iterator
does not raise, it just executes zero times. You get an empty result and no
error.

`StopIteration` here is only visible because `next()` was called directly.
In a `for` loop the same exhaustion is completely silent.

In [ ]:
reader = pd.read_csv(PATH, chunksize=250)
count = sum(1 for _ in reader)
print("first pass read", count, "chunks")
print(next(reader))